In [1]:
from vnstock import Vnstock
import pandas as pd
import numpy as np

In [2]:
stock = Vnstock().stock(symbol='ACB', source='VCI')

In [3]:
mcp=stock.listing.symbols_by_group('VN30').to_list()

In [4]:
prices = pd.DataFrame()
for i in mcp:
    stock = Vnstock().stock(symbol=i, source="VCI")
    price = stock.quote.history(start='2015-01-01', end='2024-12-31', interval="1D")
    price['ticker'] = i
    price = price[['time', 'ticker', 'close']]
    price.time = pd.to_datetime(price.time)

    # Thêm cột 'year' và 'quarter'
    price['year'] = price.time.dt.year
    price['quarter'] = price.time.dt.quarter

    # Lấy ngày giao dịch cuối cùng của mỗi quý
    price = price.loc[price.groupby(['year', 'quarter']).time.idxmax()]

    # Gộp lại dữ liệu
    price = price[['year', 'quarter', 'ticker', 'close']]
    prices = pd.concat([prices, price], ignore_index=True)


In [5]:
check=prices.ticker.value_counts().reset_index()

In [6]:
check

,ticker,count
0,ACB,40
1,SSI,40
2,STB,40
3,VCB,40
4,MWG,40
5,MSN,40
6,MBB,40
7,HPG,40
8,VIC,40
9,VNM,40


In [7]:
check=check[check['count']==40]

In [8]:
mcp=check.ticker.to_list()

In [9]:
prices=prices[prices['ticker'].isin(check['ticker'])]

In [10]:
prices

,year,quarter,ticker,close
0,2015,1,ACB,2.55
1,2015,2,ACB,3.27
2,2015,3,ACB,3.10
3,2015,4,ACB,3.16
4,2016,1,ACB,2.92
...,...,...,...,...
984,2023,4,VNM,61.80
985,2024,1,VNM,62.60
986,2024,2,VNM,60.66
987,2024,3,VNM,67.17


In [11]:
prices = pd.DataFrame()
for i in mcp:
    stock = Vnstock().stock(symbol=i, source="VCI")
    price = stock.quote.history(start='2014-01-01', end='2025-03-31', interval="1D")
    price['ticker'] = i
    price = price[['time', 'ticker', 'close']]
    price.time = pd.to_datetime(price.time)

    # Thêm cột 'year' và 'quarter'
    price['year'] = price.time.dt.year
    price['quarter'] = price.time.dt.quarter

    # Lấy ngày giao dịch cuối cùng của mỗi quý
    price = price.loc[price.groupby(['year', 'quarter']).time.idxmax()]

    # Gộp lại dữ liệu
    price = price[['year', 'quarter', 'ticker', 'close']]
    prices = pd.concat([prices, price], ignore_index=True)


In [12]:
prices['close_quarter_after'] = prices.groupby('ticker')['close'].shift(-1)

# Tính forward return
prices['forward_return'] = (prices['close_quarter_after'] / prices['close']) - 1

In [13]:
prices

,year,quarter,ticker,close,close_quarter_after,forward_return
0,2014,1,ACB,2.50,2.35,-0.060000
1,2014,2,ACB,2.35,2.32,-0.012766
2,2014,3,ACB,2.32,2.35,0.012931
3,2014,4,ACB,2.35,2.55,0.085106
4,2015,1,ACB,2.55,3.27,0.282353
...,...,...,...,...,...,...
713,2024,1,SHB,9.48,9.48,0.000000
714,2024,2,SHB,9.48,9.55,0.007384
715,2024,3,SHB,9.55,8.90,-0.068063
716,2024,4,SHB,8.90,12.00,0.348315


In [14]:
prices=prices[(prices['year']>=2015)&(prices['year']<=2024)]

In [15]:
prices

,year,quarter,ticker,close,close_quarter_after,forward_return
4,2015,1,ACB,2.55,3.27,0.282353
5,2015,2,ACB,3.27,3.10,-0.051988
6,2015,3,ACB,3.10,3.16,0.019355
7,2015,4,ACB,3.16,2.92,-0.075949
8,2016,1,ACB,2.92,3.03,0.037671
...,...,...,...,...,...,...
712,2023,4,SHB,8.98,9.48,0.055679
713,2024,1,SHB,9.48,9.48,0.000000
714,2024,2,SHB,9.48,9.55,0.007384
715,2024,3,SHB,9.55,8.90,-0.068063


In [16]:
companies_size=pd.DataFrame()
for i in mcp:
  stock = Vnstock().stock(symbol=i,source="VCI")
  balance_sheet=stock.finance.balance_sheet(period='quarter',lang='vi',dropna=True)
  balance_sheet=balance_sheet[balance_sheet['Năm']>=2015]
  balance_sheet=balance_sheet[['CP','Năm','Kỳ','TỔNG CỘNG TÀI SẢN (đồng)','NỢ PHẢI TRẢ (đồng)','VỐN CHỦ SỞ HỮU (đồng)']]
  balance_sheet['size']=np.log(balance_sheet['TỔNG CỘNG TÀI SẢN (đồng)'])
  balance_sheet['de']=balance_sheet['NỢ PHẢI TRẢ (đồng)']/balance_sheet['VỐN CHỦ SỞ HỮU (đồng)']
  balance_sheet=balance_sheet[['CP','Năm','Kỳ','size','de']]
  balance_sheet.columns=['ticker','year','quarter','size','de']
  companies_size=pd.concat([companies_size,balance_sheet])

In [17]:
companies_size

,ticker,year,quarter,size,de
0,ACB,2025,1,34.424122,9.240355
1,ACB,2024,4,34.392600,9.352125
2,ACB,2024,3,34.286967,8.858604
3,ACB,2024,2,34.276994,9.290830
4,ACB,2024,1,34.220357,8.724440
...,...,...,...,...,...
36,SHB,2016,1,32.947331,16.631796
37,SHB,2015,4,32.952882,17.193160
38,SHB,2015,3,32.842193,15.551965
39,SHB,2015,2,32.809385,15.378448


In [18]:
ratios=pd.DataFrame()
for i in mcp:
  stock = Vnstock().stock(symbol=i,source="VCI")
  ratio=stock.finance.ratio(period='quarter', lang='vi', dropna=True)
  ratio=ratio.droplevel(0,1)
  ratio=ratio[ratio['Năm']>=2015]
  ratio=ratio[['CP','Năm','Kỳ','ROE (%)','ROA (%)']]
  ratio.columns=['ticker','year','quarter','roe','roa']
  ratios=pd.concat([ratios,ratio])

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
Down

In [19]:
ratios

,ticker,year,quarter,roe,roa
0,ACB,2025,1,0.204361,0.020059
1,ACB,2024,4,0.217459,0.021215
2,ACB,2024,3,0.217115,0.021717
3,ACB,2024,2,0.228530,0.022928
4,ACB,2024,1,0.229671,0.023211
...,...,...,...,...,...
36,SHB,2016,1,0.078939,0.004589
37,SHB,2015,4,0.000000,0.000000
38,SHB,2015,3,0.000000,0.000000
39,SHB,2015,2,0.074362,0.004732


In [20]:
data=pd.merge(prices,companies_size,on=['ticker','year','quarter'])
data=pd.merge(data,ratios,on=['ticker','year','quarter'])

In [21]:
data

,year,quarter,ticker,close,close_quarter_after,forward_return,size,de,roe,roa
0,2015,1,ACB,2.55,3.27,0.282353,32.863793,13.772336,0.000000,0.000000
1,2015,2,ACB,3.27,3.10,-0.051988,32.875391,14.353217,0.000000,0.000000
2,2015,3,ACB,3.10,3.16,0.019355,32.893695,14.298959,0.000000,0.000000
3,2015,4,ACB,3.16,2.92,-0.075949,32.936597,14.754160,0.000000,0.000000
4,2016,1,ACB,2.92,3.03,0.037671,32.977499,15.016311,0.083376,0.005341
...,...,...,...,...,...,...,...,...,...,...
635,2023,4,SHB,8.98,9.48,0.055679,34.077415,11.638862,0.157517,0.012490
636,2024,1,SHB,9.48,9.48,0.000000,34.062584,10.679818,0.155816,0.012824
637,2024,2,SHB,9.48,9.55,0.007384,34.123051,10.762857,0.155879,0.012975
638,2024,3,SHB,9.55,8.90,-0.068063,34.165372,11.347513,0.145429,0.012022


In [22]:
data.to_csv('Financial_ratios.csv')

In [ ]:
prices=pd.DataFrame()
for i in mcp:
  stock = Vnstock().stock(symbol=i,source="VCI")
  price = stock.quote.history(start='2015-01-01', end='2024-12-31', interval="1D")
  price['ticker'] = i
  price = price[['time', 'ticker', 'close']]
  price=price.drop_duplicates(subset='time',keep='first')
  prices=pd.concat([prices,price])

In [ ]:
prices['returns']=prices.groupby('ticker')['close'].pct_change()

In [ ]:
prices.dropna(inplace=True)

In [ ]:
prices.time = pd.to_datetime(prices.time)


prices['year'] = prices.time.dt.year
prices['quarter'] = prices.time.dt.quarter

In [ ]:
prices.to_csv('prices.csv')